# AksharaPiece on Kaggle: repeat runs with new seeds

The first training run found AksharaPiece better than plain BPE (Transformer −0.82% bits per byte, Mamba −0.15%) with one random seed each. This notebook repeats the same four models with **new random seeds**, so we can tell a real effect from luck:

| | Plain BPE 16k | AksharaPiece BPE 16k |
|---|---|---|
| **Transformer** (8 layers) | seeds 1, 2 | seeds 1, 2 |
| **Mamba-2** (16 layers) | seeds 1, 2 | seeds 1, 2 |

Everything else is identical to the first run: same data, same tokenizers (the notebook checks this), same model sizes. It also

- uses **float16** (native on the T4), which should be faster than the first run,
- records the loss on every validation line for a **paired significance test**,
- saves every trained model (in `models/` in the Output) for later downstream tasks.

**Before you save and run**

1. **Turn on the GPU.** In the right-hand panel open **Session options → Accelerator → GPU T4 x2**. Kaggle needs a verified phone number for GPUs. Two GPUs let two models train at once (P100 also works, about twice as slow).
2. **Internet → On.**
3. **Hugging Face token** (recommended, avoids slow or stalled downloads): **Add-ons → Secrets**, make sure `HF_TOKEN` is listed and switched on for this notebook.
4. **Check it took effect:** run the first three code cells by hand. The setup cell must print `GPUs: ['Tesla T4', 'Tesla T4']`. If it prints `none`, the notebook stops on purpose: fix the accelerator first.
5. Then **Save Version → Save & Run All (Commit)**. The version page should show **Accelerator: GPU T4 x2**. Stop the editor session afterwards so it does not use GPU hours while idle.

**Run time and GPU quota.** 8 runs, two at a time. The first run took about 4.5 hours for 4 runs; float16 should make each run faster, so expect **about 4–9 hours**. Two GPUs count double against Kaggle's 30 free GPU hours per week, so check your remaining quota first (profile picture → **Settings** → quota): you need roughly 9–18 hours. If you have less, set `SEEDS = [1]` (half the time). The notebook never starts a run that could not finish before Kaggle's 12-hour limit, and saves every finished run immediately.

In [ ]:
# ---- Settings ----
RAW_MB = 1500         # Nepali text to download from IndicCorp v2 (MB); the file is 14.8 GB in total
HELDOUT_MB = 20       # held-out text, never used for training
TOKENIZER_MB = 100    # text for training the tokenizers
TRAIN_MB = 600        # text each language model trains on (identical bytes for every run)
VAL_MB = 5            # held-out text used to compute bits per byte
VOCABS = [16000]
SEEDS = [1, 2]        # seed 0 was the first run; add 0 here to rerun it in float16 as well
ARCHS = ["mamba", "transformer"]
SEQ_LEN = 1024
SAVE_MODELS = True    # float16 weights (~70 MB each) go to the Output panel under models/
KAGGLE_LIMIT_H = 12.0 # Kaggle stops a run after 12 hours; no run starts unless it can finish before that
ALLOW_CPU = False     # keep False: without a GPU the notebook stops instead of running for days

## Setup
Unpacks the project code (embedded in this notebook) and checks the GPUs.

In [ ]:
import base64, io, json, os, subprocess, sys, time, zipfile

ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = "/kaggle/working" if ON_KAGGLE else os.getcwd()          # saved by Kaggle: keep it small
WORK = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else ("/tmp/aksharapiece_work" if ON_KAGGLE else os.getcwd())
os.makedirs(WORK, exist_ok=True)                                # large data lives here, not saved as output
RUNS_OUT = os.path.join(OUT, "finished_runs")                   # each finished run is copied here at once
os.makedirs(RUNS_OUT, exist_ok=True)
T0 = time.time()
os.chdir(WORK)
CODE_ZIP_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP_B64))).extractall(WORK)   # always refresh the code
PROJECT = os.path.join(WORK, "aksharapiece")
os.chdir(PROJECT)
for p in (PROJECT, os.path.join(PROJECT, "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = PROJECT


def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")


import torch
N_GPU = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "none")
print("large data in", WORK, "| saved output in", OUT)
if N_GPU == 0 and not ALLOW_CPU:
    raise RuntimeError("No GPU found. Open Session options -> Accelerator -> GPU T4 x2, then run again. "
                       "(Stopping now so the run does not take days on the CPU.)")

In [ ]:
run("python -m pytest -q tests")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception:
    print("No HF_TOKEN secret: downloading without a token (may be slower).")

## Step 1. Download and clean Nepali text
Streams the first `RAW_MB` of the IndicCorp v2 Nepali file (CC-0), then normalizes, filters, deduplicates and splits it into held-out, tokenizer and pretraining text.

In [ ]:
import requests


def download_prefix(url, dest, limit, token=None, attempts=8):
    """Download the first `limit` bytes of a large file with HTTP range requests.
    Prints progress every 100 MB, resumes after a stall or dropped connection, and ends the file
    at a complete line."""
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    t0, next_report = time.time(), (size // 100_000_000 + 1) * 100_000_000
    for attempt in range(1, attempts + 1):
        if size >= limit:
            break
        headers = {"Range": f"bytes={size}-{limit - 1}"}
        if token:
            headers["Authorization"] = f"Bearer {token}"
        try:
            with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as r:
                if r.status_code not in (200, 206):
                    raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
                if r.status_code == 200 and size:      # server ignored the range: start over
                    size = 0
                with open(dest, "ab" if size else "wb") as out:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        out.write(chunk)
                        size += len(chunk)
                        if size >= next_report:
                            mb_s = size / 1e6 / max(1e-9, time.time() - t0)
                            print(f"  {size / 1e6:,.0f} MB downloaded ({mb_s:.1f} MB/s)", flush=True)
                            next_report += 100_000_000
                        if size >= limit:
                            break
        except Exception as e:
            print(f"  attempt {attempt}: {type(e).__name__}: {str(e)[:150]} - resuming in 10 s", flush=True)
            time.sleep(10)
    if size < limit * 0.99:
        raise RuntimeError(f"download incomplete: {size / 1e6:.0f} MB of {limit / 1e6:.0f} MB")
    with open(dest, "rb+") as f:                        # cut to the last complete line
        f.seek(max(0, size - 1_000_000))
        tail_bytes = f.read()
        f.truncate(size - len(tail_bytes) + tail_bytes.rfind(b"\n") + 1)
    print(f"saved {os.path.getsize(dest) / 1e6:,.0f} MB in {time.time() - t0:.0f}s", flush=True)


RAW_FILE = "data/raw/indiccorp_ne_sample.txt"
os.makedirs("data/raw", exist_ok=True)
if not (os.path.exists(RAW_FILE) and os.path.getsize(RAW_FILE) >= RAW_MB * 0.99e6):
    from huggingface_hub import get_token, hf_hub_url
    download_prefix(hf_hub_url("ai4bharat/IndicCorpV2", "data/ne.txt", repo_type="dataset"),
                    RAW_FILE, RAW_MB * 1_000_000, token=get_token())

if not os.path.exists("data/clean/stats.json"):
    run(f"python scripts/clean.py --inputs {RAW_FILE} --out data/clean --heldout-mb {HELDOUT_MB} "
        f"--tok-mb {TOKENIZER_MB} --pretrain-mb 100000")
stats = json.load(open("data/clean/stats.json"))
print(f"kept {stats['kept']:,} lines; MB written: {stats['written_mb']}")
assert stats["written_mb"]["pretrain.txt"] >= TRAIN_MB * 0.98, "not enough pretraining text: raise RAW_MB or lower TRAIN_MB"

## Step 2. Train the tokenizers
Plain BPE and AksharaPiece BPE, same trainer, same text, same vocabulary size. AksharaPiece keeps the syllables that cover 99.9% of occurrences as whole symbols (99.5% below 16k).

In [ ]:
run(f"python scripts/train_tokenizers.py --train-file data/clean/tokenizer_train.txt --train-mb {TOKENIZER_MB} "
    f"--vocab {' '.join(map(str, VOCABS))} --out data/tok")

## Step 3. Tokenize the training and validation text
Every tokenizer encodes the identical bytes. The table shows how many tokens each needs.

In [ ]:
TOKS = []
for v in VOCABS:
    k = f"bpe_{v // 1000}k"
    TOKS += [(f"plain_{k}", f"hf:data/tok/plain_{k}.json"),
             (f"akshara_{k}", f"aksharapiece:data/tok/akshara_{k}.aksharapiece.json")]
tok_args = " ".join(f"--tok {n}={s}" for n, s in TOKS)
run(f"python scripts/prepare_tokens.py --train-file data/clean/pretrain.txt --train-mb {TRAIN_MB} "
    f"--val-file data/clean/heldout.txt --val-mb {VAL_MB} --out data/tokens --workers 4 {tok_args}")

# Same data as the first run? (token counts from the first Kaggle training run)
FIRST_RUN = {"plain_bpe_16k": 47_990_357, "akshara_bpe_16k": 48_758_535}
for n, _ in TOKS:
    got = json.load(open(f"data/tokens/{n}/meta.json"))["train_tokens"]
    if n in FIRST_RUN:
        same = got == FIRST_RUN[n]
        print(f"{n}: {got:,} train tokens -> {'identical to the first run' if same else 'DIFFERENT from the first run'}")

## Step 4. Train the language models
Runs go to the GPUs in parallel (one per GPU), Mamba runs first because they are slower. Progress is printed every few minutes; each run's full log is saved in `results/lm/logs/`. Finished runs are skipped if you run this cell again.

In [ ]:
import shutil

os.makedirs("results/lm/logs", exist_ok=True)
queue = [(n, a, s) for s in SEEDS for a in ARCHS for n, _ in TOKS
         if not os.path.exists(f"results/lm/{n}__{a}__s{s}.json")]
print(f"{len(queue)} runs to do:", queue)
# expected minutes per run, updated from finished runs (first-run timings as a cautious start)
EST_MIN = {"mamba": 150.0, "transformer": 120.0}


SHRINK = {}  # (tok, arch) -> how many times the batch was halved after running out of GPU memory


def cmd_for(tok, arch, seed):
    # Mamba uses more memory in its scan: half the batch, two accumulation steps (same tokens per step)
    batch, accum = (8, 2) if arch == "mamba" else (16, 1)
    k = SHRINK.get((tok, arch, seed), 0)
    batch, accum = max(1, batch >> k), accum << k     # same tokens per step, less memory
    return (f"python scripts/train_lm.py --tokens data/tokens/{tok} --arch {arch} --out results/lm "
            f"--seq-len {SEQ_LEN} --batch {batch} --grad-accum {accum} --seed {seed} --eval-every 500"
            + (f" --save-model --model-dir {os.path.join(OUT, 'models')}" if SAVE_MODELS else ""))


def tail(path, n=1):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 3000))
            lines = [l for l in f.read().decode("utf-8", "replace").splitlines() if l.strip()]
            return lines[-n:]
    except FileNotFoundError:
        return []


slots = list(range(max(1, N_GPU)))
running, failed, skipped, t_start, last_print = {}, [], [], time.time(), 0
while queue or running:
    for g in slots:
        if g not in running and queue:
            tok, arch, seed = queue[0]
            left_min = (KAGGLE_LIMIT_H * 3600 - (time.time() - T0)) / 60
            if EST_MIN[arch] * 1.15 + 15 > left_min:       # would not finish before Kaggle's limit
                skipped.append(queue.pop(0))
                print(f"    not starting {tok}/{arch}/seed {seed}: ~{EST_MIN[arch]:.0f} min needed, {left_min:.0f} min left", flush=True)
                continue
            queue.pop(0)
            log = f"results/lm/logs/{tok}__{arch}__s{seed}.log"
            env = dict(os.environ)
            if N_GPU:
                env["CUDA_VISIBLE_DEVICES"] = str(g)
            p = subprocess.Popen(cmd_for(tok, arch, seed), shell=True, env=env,
                                 stdout=open(log, "w"), stderr=subprocess.STDOUT)
            running[g] = (p, tok, arch, seed, log, time.time())
            print(f"[{(time.time() - t_start) / 60:6.1f} min] started {tok} / {arch} / seed {seed} on GPU {g}", flush=True)
    for g, (p, tok, arch, seed, log, t_run) in list(running.items()):
        if p.poll() is not None:
            ok = p.returncode == 0
            print(f"[{(time.time() - t_start) / 60:6.1f} min] {'finished' if ok else 'FAILED'} {tok} / {arch} / seed {seed}: "
                  f"{tail(log)[-1] if tail(log) else ''}", flush=True)
            if ok:
                EST_MIN[arch] = (time.time() - t_run) / 60       # real duration of this kind of run
                base = f"results/lm/{tok}__{arch}__s{seed}"
                for f in (base + ".json", base + ".lines.npy", log):
                    if os.path.exists(f):
                        shutil.copy(f, RUNS_OUT)
            if not ok:
                key = (tok, arch, seed)
                if "out of memory" in " ".join(tail(log, 40)).lower() and SHRINK.get(key, 0) < 3:
                    SHRINK[key] = SHRINK.get(key, 0) + 1
                    queue.insert(0, key)
                    print(f"    out of GPU memory: retrying {tok}/{arch}/seed {seed} with half the batch", flush=True)
                else:
                    failed.append(key)
                    print("\n".join(tail(log, 25)))
            del running[g]
    if time.time() - last_print > 300 and running:
        last_print = time.time()
        for g, (p, tok, arch, seed, log, t_run) in running.items():
            print(f"    GPU {g} {tok}/{arch}/s{seed}: {(tail(log) or [''])[-1].strip()}", flush=True)
    time.sleep(10)
print("all runs done" + (f"; failed: {failed}" if failed else "") + (f"; not started (time): {skipped}" if skipped else ""))

## Step 5. Results
**Bits per byte** is the headline: lower means the model predicts Nepali text better. Compare plain vs AksharaPiece within each architecture.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

run("python scripts/summarize_lm.py --runs results/lm --out results/lm_summary")
display(Image("results/lm_summary.png"))

display(Markdown(open("results/lm_summary.md").read()))
for v in VOCABS:
    k = f"bpe_{v // 1000}k"
    run(f"python scripts/significance.py --runs results/lm --line-bytes data/tokens/plain_{k}/val_line_bytes.bin "
        f"--a plain_{k} --b akshara_{k} --out results/significance_{k}")
    display(Markdown(open(f"results/significance_{k}.md").read()))

## Step 6. Save the results
`aksharapiece_lm_results.zip` (in the **Output** panel) holds the summary table and chart, the significance test, every run's JSON, per-line scores and log, and the tokenizers. **Send this zip back** (it is small). The trained models stay in the Output panel under `models/`; do not send those.

In [ ]:
import shutil

bundle = os.path.join(WORK, "lm_bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree("results", os.path.join(bundle, "results"))
shutil.copytree("data/tok", os.path.join(bundle, "tokenizers"))
shutil.copy(f"data/tokens/plain_bpe_{VOCABS[0] // 1000}k/val_line_bytes.bin", bundle)
for f in ("data/clean/stats.json",):
    shutil.copy(f, bundle)
for n, _ in TOKS:
    shutil.copy(f"data/tokens/{n}/meta.json", os.path.join(bundle, f"tokens_meta_{n}.json"))
shutil.make_archive(os.path.join(OUT, "aksharapiece_lm_results"), "zip", bundle)
shutil.rmtree(bundle)
print("Saved", os.path.join(OUT, "aksharapiece_lm_results.zip"))

## How to read the results

- **Significance table:** for each seed, the 95% interval of the difference. If the whole interval is below zero, AksharaPiece is reliably better on that seed. The "mean" row combines seeds.
- **Transformer vs Mamba gap:** research question 3 asks whether the tokenizer matters more for Mamba.
- **Bytes per token:** AksharaPiece needs slightly more tokens for the same text; bits per byte already accounts for that.
- Next steps after this run: repeat with 2 more seeds, add the 8k vocabulary, and run the downstream tasks.